In [ ]:
import torch

# Simulate a gradient tensor containing three 3x4 matrices
gradients = torch.full((3, 3, 4), 0.5)

# Inject an exploding gradient into the very first matrix
gradients[0, 0, 0] = 10.0
gradients[0, 0, 1] = 8.0

print("=== BATCH BEFORE CLIPPING ===")
for i in range(3):
    print(f"Matrix {i+1}:")
    print(gradients[i])
    print()

=== BATCH BEFORE CLIPPING ===
Matrix 1:
tensor([[10.0000,  8.0000,  0.5000,  0.5000],
        [ 0.5000,  0.5000,  0.5000,  0.5000],
        [ 0.5000,  0.5000,  0.5000,  0.5000]])

Matrix 2:
tensor([[0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000]])

Matrix 3:
tensor([[0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000]])



In [ ]:
# Calculate the original Global L2 Norm
# flatten the tensor to calculate the norm across all four matrices
original_norm = torch.linalg.vector_norm(gradients)
print(f"\nOriginal Global Norm: {original_norm:.2f}")


Original Global Norm: 13.13


Because the L2 norm squares everything before adding it up, it is highly sensitive to massive outliers. If even a single parameter in your massive 3x4 matrix explodes, it will completely dominate the total sum. This causes the total L2 norm to spike past your max_norm limit, immediately triggering the clipping mechanism to rescue your model and scale everything down to safety.

In [ ]:
# Apply Gradient Clipping
max_norm = 5.0
clip_coef = max_norm / (original_norm + 1e-6) # 1e-6 prevents division by zero

# If the original norm is bigger than 5.0, scale everything down
if clip_coef < 1.0:
    clipped_gradients = gradients * clip_coef
else:
    clipped_gradients = gradients

print("\n=== BATCH AFTER CLIPPING ===")
for i in range(3):
    print(f"Matrix {i+1}:")
    print(clipped_gradients[i]) # clipping will always happen before optimizer updates weights
    print()

new_norm = torch.linalg.vector_norm(clipped_gradients)
print(f"\nNew Global Norm: {new_norm:.2f}")


=== ENTIRE BATCH AFTER CLIPPING (Max Norm = 5.0) ===
Matrix 1:
tensor([[3.8069, 3.0455, 0.1903, 0.1903],
        [0.1903, 0.1903, 0.1903, 0.1903],
        [0.1903, 0.1903, 0.1903, 0.1903]])

Matrix 2:
tensor([[0.1903, 0.1903, 0.1903, 0.1903],
        [0.1903, 0.1903, 0.1903, 0.1903],
        [0.1903, 0.1903, 0.1903, 0.1903]])

Matrix 3:
tensor([[0.1903, 0.1903, 0.1903, 0.1903],
        [0.1903, 0.1903, 0.1903, 0.1903],
        [0.1903, 0.1903, 0.1903, 0.1903]])


New Global Norm: 5.00
